# JSCC neuronal pour processus ponctuels sur canal à jitter — v2

Reprise complète de `hawkes_jscc.ipynb` sur la base du même énoncé
(`student_project_point_process_jscc.pdf`) : même canal à jitter gaussien, même
rate $R = T_s/T_c = 1$, même contrainte $N_X = N_S$, même distorsion (éq. 7-8).

## Pourquoi une v2 : le diagnostic de la v1

Dans la v1, la contrainte $N_X = N_S$ était **déjà correcte** (top-N par rang et
`gather_at_spikes` la garantissaient à 100 %). Le vrai problème était ailleurs.

En comparant les chiffres de la v1 à un **décodeur aveugle** — un décodeur qui
ignore totalement $Y$ et sort $\mathbb{E}[T_i \mid N, i]$ — on obtient :

| source | D aveugle | v1 `jscc` à $\sigma/\mu = 4$ | v1 `jscc` à $\sigma/\mu = 0.1$ |
|---|---|---|---|
| poisson | 0.0185 | 0.0305 | — |
| gamma k=4 | 0.0051 | 0.0130 | — |
| hawkes | 0.0341 | 0.0498 | 0.0498 |

La distorsion de la v1 était **plate** sur quatre décades de bruit et du même
ordre que le décodeur aveugle : le système ne transmettait aucune information
temporelle. Le « gain de 4 à 15× sur `uncoded` » à fort bruit n'était pas un gain
JSCC, seulement le fait qu'un prédicteur a priori bat la transmission brute quand
le SNR est mauvais.

**Cause racine : l'identité n'était pas représentable.** Avec
$\Delta t_i = \mathrm{softplus}(\text{head}(u_i))$ et $u$ le potentiel d'un LIF à
fuite, l'écart $g_i$ est encodé dans $u \approx a\beta^{g_i/\Delta} + b$, donc
exponentiellement. Pour recopier l'entrée il faudrait un logarithme, et `head` est
linéaire. La meilleure fonction de la famille est $\Delta t \approx$ constante —
exactement la solution trouvée. Ce n'était pas un problème d'optimisation.

## Les cinq changements

1. **Paramétrisation en déplacement** (Step 3 du PDF, « encoder displacement
   constraint ») : $g^X_i = g_i\,e^{a_i}$ et $\hat{S}_i = Y_{(i)} + c_i$, avec les
   têtes de lecture initialisées à zéro. **À l'initialisation, le système *est*
   `uncoded`** — il ne peut que s'améliorer, et le doute « le décodeur devrait au
   moins égaler uncoded » disparaît par construction.
2. **SNN événementiel** : le LIF est déroulé événement par événement, la fuite
   entre deux événements étant $e^{-g_i/\tau}$, solution *exacte* de l'EDO. C'est
   le même neurone qu'en temps discret, sans quantifier le temps. Conséquences :
   plus de collisions de bins (la v1 en avait 26 % à 58 % **dans la source**), plus
   de déroulement à 128 pas, et $N$ conservé trivialement.
3. **Baseline aveugle partout** : chaque tableau et chaque figure porte la ligne
   $D_{\text{aveugle}}$. C'est le chiffre qui dit si le système transmet
   quoi que ce soit.
4. **Pas de troncature `min(N_S, N_Ŝ)`** : les trois comptes sont égaux par
   construction, donc la distorsion est l'éq. 7 exacte. Dans la v1 la
   normalisation par `min()` *récompensait* la perte d'événements.
5. **Step 3 avec l'optimum numérique** : pour le modèle à deux événements, on
   calcule l'encodeur optimal (fonction monotone libre sur une grille) et le
   décodeur MMSE exact. On compare le réseau à l'**optimum**, pas à `uncoded`.

## Ce que le notebook montre

- Le gain JSCC existe et il est **grand** : dans le modèle à deux événements,
  l'encodeur optimal fait 2 à 10× mieux que le meilleur décodeur seul.
- Le mécanisme est une **dilatation** : l'encodeur étale la distribution des
  écarts sur toute la fenêtre $[0, T_c]$ (pente locale $\approx 6.6$ sur Gamma),
  ce qui augmente le SNR effectif du canal de timing à coût d'événements constant.
- Dans le problème complet à $N$ événements, le gain est **beaucoup plus petit**
  (10-20 %) — et c'est cohérent : la source remplit déjà la fenêtre
  ($\sum_i g_i \approx T_c$), donc l'encodeur ne peut que *réallouer*, pas dilater.

**Durée d'exécution complète : de l'ordre de 30 minutes sur 4 cœurs CPU** (≈ 15 min
pour l'expérience 1, 7 min pour l'expérience 2, 6 min pour le Step 3). Les constantes
`STEPS`, `SEEDS` et `SIGMA_GRID` de la cellule de configuration permettent de réduire :
`STEPS = 200, SEEDS = [0]` fait tourner le tout en 5 minutes environ, suffisamment pour
vérifier que la chaîne fonctionne (les écarts entre `decoder` et `jscc` deviennent
alors trop bruités pour être interprétés).

> **Statut de validation.** Les briques de calcul — sources, `EventLIF`, `ResidualSystem`,
> `distortion`, `train`, `evaluate`, `Grid`, `optimal_encoder`, `train_tiny` — ont été
> exécutées et vérifiées. Les cellules de figures et de tableaux ont été relues mais pas
> exécutées : si l'une casse, c'est de l'affichage, pas du calcul.

## 1. Imports et configuration

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, replace
import math, time, json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(4)
DEVICE = "cpu"


@dataclass
class Config:
    # ------- source / canal -------------------------------------------------
    Ts: float = 1.0              # durée du bloc source ; Tc = Ts donc R = 1
    mean_gap: float = 1.0 / 9.0  # échelle de temps de la source (µ)
    sigma_ratio: float = 1.0     # sigma_J / mean_gap  (le seul paramètre de bruit)

    # ------- réseau ----------------------------------------------------------
    hidden: int = 32             # largeur de l'unique couche LIF
    n_in: int = 2                # features par événement : [g_i/µ, T_i/Tc]
    spike_alpha: float = 2.0     # pente du surrogate arctan
    a_max: float = math.log(4.0) # dilatation max d'un écart par l'encodeur (×4 / ÷4)
    c_max_ratio: float = 3.0     # déplacement max du décodeur, × max(µ, sigma_J)

    # ------- optimisation ----------------------------------------------------
    steps: int = 600
    batch: int = 128
    lr: float = 3e-3
    seed: int = 0

    @property
    def Tc(self) -> float:
        return self.Ts                      # R = Ts/Tc = 1

    @property
    def sigma_J(self) -> float:
        return self.sigma_ratio * self.mean_gap


CFG = Config()

# --- constantes d'exécution (réduire pour un test rapide) --------------------
STEPS       = 600
SEEDS       = [0, 1]
SIGMA_GRID  = [0.3, 1.0, 3.0]
N_TRAIN, N_EVAL = 20_000, 8_000
MODES = ("uncoded", "decoder", "jscc")

print(CFG)
print(f"sigma_J = {CFG.sigma_J:.4f}   (mean_gap = {CFG.mean_gap:.4f})")

### Conventions de tracé

Même palette que les notebooks précédents pour que les trois se lisent ensemble.
`C_BLIND` (gris) est nouveau : c'est la ligne du décodeur aveugle.

In [ ]:
C_UNCODED, C_DECODER, C_JSCC = "#2a78d6", "#eb6834", "#1baf7a"
C_BLIND, C_OPT = "#8a8983", "#b0342d"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8983", "#e4e3df", "#fcfcfb"
MODE_STYLE = {"uncoded": (C_UNCODED, "o"), "decoder": (C_DECODER, "s"),
              "jscc": (C_JSCC, "^")}


def style(ax, xlabel, ylabel, title, subtitle=None):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9, length=3, color=GRID)
    ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    ax.set_ylabel(ylabel, color=INK2, fontsize=10)
    ax.set_title(title, color=INK, fontsize=12, loc="left", pad=26 if subtitle else 8)
    if subtitle:
        ax.text(0.0, 1.02, subtitle, transform=ax.transAxes, color=MUTED,
                fontsize=9, va="bottom")


def legend(ax, loc="best"):
    leg = ax.legend(frameon=False, fontsize=9, loc=loc)
    for t in leg.get_texts():
        t.set_color(INK2)


def show_table(rows, cols=None, title=""):
    cols = cols or list(rows[0].keys())
    w = {c: max(len(c), 11) for c in cols}
    if title:
        print(title)
    print(" | ".join(c.rjust(w[c]) for c in cols))
    print("-+-".join("-" * w[c] for c in cols))
    for r in rows:
        cells = []
        for c in cols:
            v = r.get(c, "")
            if isinstance(v, float):
                v = f"{v:.4e}" if (v != 0 and abs(v) < 1e-3) else f"{v:.4f}"
            cells.append(str(v).rjust(w[c]))
        print(" | ".join(cells))

## 2. Sources — représentation événementielle

**Changement de représentation par rapport à la v1.** Un bloc n'est plus un raster
binaire de `T_bins` pas mais une liste de temps continus, paddée à droite et
accompagnée d'un masque : `(t (B, N_max), mask (B, N_max))`. Il n'y a plus aucune
quantification du temps nulle part dans le pipeline.

Pourquoi ça compte : la v1 mesurait `frac_source_collision` entre 0.26 et 0.58 —
dans plus d'un bloc sur quatre, **la source elle-même** perdait au moins un
événement à la rastérisation, avant tout encodage. La cellule de diagnostic plus
bas rechiffre cet effet pour justifier le changement.

Trois familles, correspondant aux trois étapes du PDF :

- **Poisson homogène** (Step 1) : renouvellement exponentiel, la source la plus
  « aléatoire » ;
- **renouvellement Gamma** (Step 2, éq. 10) : $W_i \sim \Gamma(k, \mu/k)$, à moyenne
  fixée, $k$ balayé dans $\{1, 2, 4, 8\}$ ;
- **Hawkes** à noyau exponentiel (extension de la v1) : $\lambda(t) = \mu_0 +
  \alpha\sum_{T_i<t} e^{-\beta(t-T_i)}$, mémoire multi-événements et rafales.

Dans les trois cas la fenêtre $[0, T_s)$ est fixe et $N_S$ est **aléatoire** : c'est
une conséquence du tirage, jamais un paramètre.

In [ ]:
def _pack(blocks):
    """Liste de tableaux de temps -> (t (B,N_max), mask (B,N_max))."""
    N = max(max(len(b) for b in blocks), 1)
    t = np.zeros((len(blocks), N), dtype=np.float32)
    m = np.zeros((len(blocks), N), dtype=bool)
    for i, b in enumerate(blocks):
        t[i, :len(b)] = b
        m[i, :len(b)] = True
    return torch.from_numpy(t).to(DEVICE), torch.from_numpy(m).to(DEVICE)


def gen_renewal(cfg, n_blocks, seed, k=1.0):
    """Renouvellement Gamma vectorisé, W_i ~ Gamma(k, µ/k). k = 1 -> Poisson."""
    rng = np.random.default_rng(seed)
    K = int(6 * cfg.Ts / cfg.mean_gap) + 20          # marge : P(N > K) négligeable
    T = np.cumsum(rng.gamma(k, cfg.mean_gap / k, size=(n_blocks, K)), axis=1)
    keep = T < cfg.Ts
    return _pack([T[i, keep[i]] for i in range(n_blocks)])


def _hawkes_one(mu, alpha, beta, Ts, n_burn, cap, rng):
    """Un bloc par amincissement d'Ogata (1981), avec rodage pour démarrer en
    régime stationnaire. On s'arrête dès que le temps candidat dépasse Ts."""
    t, e = 0.0, 0.0
    for _ in range(n_burn):
        M = mu + e
        tau = rng.exponential(1.0 / M); e *= np.exp(-beta * tau); t += tau
        if rng.random() <= (mu + e) / M:
            e += alpha
    t0, out = t, []
    while len(out) < cap:
        M = mu + e
        tau = rng.exponential(1.0 / M); e *= np.exp(-beta * tau); t += tau
        if t - t0 >= Ts:
            break
        if rng.random() <= (mu + e) / M:
            out.append(t - t0); e += alpha
    return np.array(out)


def gen_hawkes(cfg, n_blocks, seed, branching=0.6, decay=1.0, n_burn=40, cap=200):
    """n = alpha/beta dans (0,1) : part du taux due à l'auto-excitation.
    mu0, alpha, beta sont dérivés pour que le taux stationnaire reste 1/mean_gap."""
    rng = np.random.default_rng(seed)
    beta = 1.0 / (decay * cfg.mean_gap)
    alpha = branching * beta
    mu = (1.0 / cfg.mean_gap) * (1.0 - branching)
    return _pack([_hawkes_one(mu, alpha, beta, cfg.Ts, n_burn, cap, rng)
                  for _ in range(n_blocks)])


SOURCES = {
    "poisson":  lambda cfg, n, s: gen_renewal(cfg, n, s, k=1.0),
    "gamma_k2": lambda cfg, n, s: gen_renewal(cfg, n, s, k=2.0),
    "gamma_k4": lambda cfg, n, s: gen_renewal(cfg, n, s, k=4.0),
    "gamma_k8": lambda cfg, n, s: gen_renewal(cfg, n, s, k=8.0),
    "hawkes":   lambda cfg, n, s: gen_hawkes(cfg, n, s, branching=0.6),
}


class Dataset:
    """Jeu pré-généré. Générer les blocs une fois et échantillonner dedans évite
    de payer la simulation de la source à chaque pas d'entraînement (c'était
    l'essentiel du coût de la v1)."""

    def __init__(self, t, m):
        self.t, self.m = t, m

    def sample(self, batch, gen):
        idx = torch.randint(0, self.t.shape[0], (batch,), generator=gen)
        t, m = self.t[idx], self.m[idx]
        N = int(m.sum(1).max().item())
        return t[:, :N].clone(), m[:, :N].clone()


def gaps(t, m):
    """(B,N) temps croissants -> (B,N) écarts. g_0 = T_1 (depuis l'origine),
    g_i = T_{i+1} - T_i ensuite : c'est l'éq. 12 du PDF décalée d'un indice."""
    prev = torch.cat([torch.zeros_like(t[:, :1]), t[:, :-1]], 1)
    return (t - prev) * m.to(t.dtype)

In [ ]:
# --- statistiques de source + coût qu'aurait eu la rastérisation --------------
def source_stats(name, cfg, n=4000, seed=12345, T_bins=128):
    t, m = SOURCES[name](cfg, n, seed)
    n_s = m.sum(1).float()
    bw = cfg.Ts / T_bins
    idx = (t / bw).long()
    coll = 0
    tn, mn = idx.numpy(), m.numpy()
    for b in range(tn.shape[0]):
        v = tn[b][mn[b]]
        coll += int(len(np.unique(v)) != len(v))
    return {"source": name, "mean_n_s": float(n_s.mean()), "std_n_s": float(n_s.std()),
            "max_n_s": int(n_s.max()),
            f"frac_collision_si_T_bins_{T_bins}": coll / tn.shape[0]}


rows = [source_stats(s, CFG) for s in SOURCES]
show_table(rows, title="Statistiques des sources (N_S aléatoire, fenêtre fixe)")
print()
print("La dernière colonne est le taux de blocs qui PERDRAIENT au moins un événement")
print("si on rastérisait à T_bins=128, comme le faisait la v1. C'est la raison du")
print("passage à une représentation événementielle : ici, ce taux est 0 par construction.")

In [ ]:
def fig_sources(cfg, names=("poisson", "gamma_k8", "hawkes"), n_blocks=12, seed=3):
    fig, axes = plt.subplots(len(names), 1, figsize=(8.0, 2.4 * len(names)),
                             facecolor=SURFACE, sharex=True)
    for ax, name in zip(np.atleast_1d(axes), names):
        t, m = SOURCES[name](cfg, n_blocks, seed)
        times = [t[i][m[i]].numpy() for i in range(n_blocks)]
        ax.eventplot(times, colors=INK, lineoffsets=np.arange(n_blocks),
                     linelengths=0.7, linewidths=1.6, zorder=5)
        ax.set_ylim(-0.8, n_blocks - 0.2); ax.set_yticks([])
        style(ax, "temps  (unités de $T_s$)", "",
              f"{name}   (N_S moyen = {np.mean([len(x) for x in times]):.1f}, "
              f"min {min(len(x) for x in times)}, max {max(len(x) for x in times)})")
    fig.tight_layout()
    return fig


fig_sources(CFG); plt.show()

## 3. Les deux baselines qui manquaient

### `uncoded` : $X = S$

Avec $\hat S = Y$ on aurait $D = \sigma_J^2$ exactement. Mais le récepteur **sait**
que les événements sont ordonnés ; trier $Y$ est donc gratuit et c'est une
projection isotone, qui ne peut que réduire la MSE. La vraie baseline non codée
est donc $D_{\text{uncoded}} \le \sigma_J^2$, et l'écart grandit avec le bruit.
C'est déjà un mini-résultat JSCC : une partie du « gain » à fort bruit vient
simplement du tri, pas du réseau.

### `blind` : le décodeur aveugle

$\hat S_i = \mathbb{E}[T_i \mid N, i]$, estimé sur les données d'entraînement. Il
ignore totalement $Y$. **Tant que $D \approx D_{\text{blind}}$, aucune information
temporelle ne traverse le système**, quelle que soit l'allure de la courbe. C'est
le test qui a révélé le problème de la v1 et il doit figurer partout.

In [ ]:
def blind_table(t, m):
    """E[T_i | N, i] estimé sur un jeu de blocs -> dict (N, i) -> valeur."""
    n = m.sum(1).cpu().numpy(); tn = t.cpu().numpy()
    acc = {}
    for b in range(tn.shape[0]):
        for i in range(int(n[b])):
            acc.setdefault((int(n[b]), i), []).append(tn[b, i])
    return {k: float(np.mean(v)) for k, v in acc.items()}


def blind_D(tab, t, m):
    """D d'un décodeur qui ignore Y et sort la table ci-dessus."""
    n = m.sum(1).cpu().numpy(); tn = t.cpu().numpy()
    ds = []
    for b in range(tn.shape[0]):
        N = int(n[b])
        if N == 0:
            continue
        ds.append(np.mean([(tn[b, i] - tab.get((N, i), tn[b, i])) ** 2
                           for i in range(N)]))
    return float(np.mean(ds))

## 4. Le SNN — LIF à temps continu, déroulé événement par événement

Un seul ingrédient spiking, la cellule LIF : seuil dur en avant (Heaviside),
gradient de substitution arctan en arrière (Neftci, Mostafa & Zenke 2019).

**Ce qui change par rapport à la v1.** Le réseau n'est plus déroulé sur une horloge
de `T_bins` pas mais sur les $N$ événements du bloc. Entre l'événement $i-1$ et
l'événement $i$, séparés de $g_i$, le potentiel décroît de $e^{-g_i/\tau}$ : c'est
la **solution exacte** de l'EDO de fuite du LIF, pas une approximation. C'est donc
le même neurone, simplement sans quantifier le temps — l'approche standard des
processus ponctuels neuronaux (Du et al. 2016 ; Mei & Eisner 2017).

Trois conséquences directes :

- $g_i$ arrive dans le calcul **explicitement**, en plus d'être implicite dans la
  fuite. L'identité devient représentable par une tête linéaire, ce qui n'était pas
  le cas en v1 (il aurait fallu inverser $\beta^{g/\Delta}$) ;
- plus de collisions de bins, ni côté source ni côté canal ;
- $N$ pas de déroulement au lieu de 128 : environ 10× plus rapide, et un chemin de
  gradient bien plus court (la cause n° 1 des difficultés d'optimisation listées
  en v1).

`s` n'est pas détaché dans le terme de reset (1re approximation SuperSpike) :
c'est la correction trouvée en v1, sans quoi `theta` ne reçoit aucun gradient dès
que le readout lit $u$ plutôt que $s$.

Le décodeur est **bidirectionnel** (deux passes LIF, avant et arrière, concaténées) :
c'est un code par bloc, le récepteur dispose de tout le bloc avant de décoder.
L'encodeur reste **causal**.

In [ ]:
class ATanSpike(torch.autograd.Function):
    """Heaviside en avant, surrogate arctan en arrière (Neftci, Mostafa & Zenke)."""

    @staticmethod
    def forward(ctx, u, alpha):
        ctx.save_for_backward(u); ctx.alpha = alpha
        return (u > 0).to(u.dtype)

    @staticmethod
    def backward(ctx, go):
        (u,) = ctx.saved_tensors; a = ctx.alpha
        return go * a / (2 * (1 + (math.pi / 2 * a * u) ** 2)), None


def spike(u, alpha=2.0):
    return ATanSpike.apply(u, alpha)


def reverse_seq(x, n):
    """Inverse chaque séquence sur ses n premiers éléments seulement (le padding
    reste à droite). L'opération est une involution sur la partie valide, donc la
    même fonction sert à inverser puis à remettre dans l'ordre."""
    N = x.shape[1]
    ar = torch.arange(N, device=x.device).unsqueeze(0)
    idx = (n.unsqueeze(1) - 1 - ar).clamp(min=0)
    if x.dim() == 3:
        idx = idx.unsqueeze(-1).expand(-1, -1, x.shape[2])
    return torch.gather(x, 1, idx)


class EventLIF(nn.Module):
    """u <- u * exp(-g_i/tau) * (1 - s) + W x_i ;  s = H(u - theta).

    tau (constante de temps, en unités de mean_gap) et theta (seuil) sont appris
    par unité. Une seule couche cachée, feed-forward : la mémoire d'un événement
    au suivant vient uniquement de la fuite du potentiel."""

    def __init__(self, n_in, hidden, alpha=2.0, tau_init=1.0):
        super().__init__()
        self.syn = nn.Linear(n_in, hidden)
        self.log_tau = nn.Parameter(torch.full((hidden,), math.log(tau_init)))
        self.theta = nn.Parameter(torch.ones(hidden))
        self.alpha, self.hidden = alpha, hidden

    def forward(self, x, dt):
        """x (B,N,n_in), dt (B,N) en unités de mean_gap -> u (B,N,H)."""
        B, N, _ = x.shape
        tau = torch.exp(self.log_tau).clamp(1e-2, 1e2)
        u = torch.zeros(B, self.hidden, device=x.device)
        s = torch.zeros_like(u)
        cur = self.syn(x)
        out = []
        for i in range(N):
            u = u * torch.exp(-dt[:, i:i + 1] / tau) * (1.0 - s) + cur[:, i]
            s = spike(u - self.theta, self.alpha)
            out.append(u)
        return torch.stack(out, 1)


class Readout(nn.Module):
    """EventLIF (uni- ou bidirectionnel) + tête linéaire initialisée à ZÉRO.
    Sortie nulle à l'initialisation -> déplacement nul -> le système démarre
    exactement sur `uncoded`."""

    def __init__(self, cfg, bidir=False):
        super().__init__()
        self.bidir = bidir
        self.fwd = EventLIF(cfg.n_in, cfg.hidden, cfg.spike_alpha)
        self.bwd = EventLIF(cfg.n_in, cfg.hidden, cfg.spike_alpha) if bidir else None
        self.head = nn.Linear(cfg.hidden * (2 if bidir else 1), 1)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)

    def forward(self, x, dt, n):
        u = self.fwd(x, dt)
        if self.bidir:
            ub = reverse_seq(self.bwd(reverse_seq(x, n), reverse_seq(dt, n)), n)
            u = torch.cat([u, ub], -1)
        return self.head(u).squeeze(-1)

## 5. Le système — paramétrisation en déplacement

$$
g^X_i = g_i\,e^{a_{\max}\tanh f_i},\qquad
X = \operatorname{cumsum}(g^X),\qquad
Y = X + Z,\qquad
\hat S_i = Y_{(i)} + c_{\max}\tanh h_i
$$

Les garanties structurelles, vraies **pour tout poids**, avant tout entraînement :

- $N_X = N_S = N_{\hat S}$ : l'encodeur produit un multiplicateur par écart
  d'entrée, ni plus ni moins ; le décodeur un déplacement par événement reçu.
  Aucune pénalité de loss, aucun top-N, aucune collision possible.
- $g^X_i > 0$ : la forme multiplicative garantit que $X$ reste strictement
  croissant, donc pas de réordonnancement côté encodeur (c'était un bug réel en v1).
- $X \subset [0, T_c]$ : après le cumul on renormalise si $\sum_i g^X_i > T_c$.
  C'est la version *sample-wise* de la contrainte, la même que $N_X = N_S$.
- $R = T_s/T_c = 1$ et $c(X) = N_X/T_c = N_S/T_s$ : rate et coût d'entrée
  identiques à `uncoded`, comme l'exige la comparaison contrôlée du PDF.
- **À l'initialisation, $X = S$ et $\hat S = Y_{(\cdot)}$** : les trois modes sont
  numériquement identiques. Vérifié dans la cellule suivante.

Le tri de $Y$ au récepteur est légitime (il sait que les événements sont ordonnés)
et il est déjà comptabilisé dans la baseline `uncoded`, donc il ne fausse pas la
comparaison.

$c_{\max}$ est proportionnel à $\max(\mu, \sigma_J)$ et non à $\mu$ seul : à fort
bruit la correction nécessaire est de l'ordre de $\sigma_J$, et un plafond fixé à
$3\mu$ bride le décodeur — testé, ça coûtait un facteur 1.5 sur la distorsion à
$\sigma_J/\mu = 3$.

In [ ]:
class ResidualSystem(nn.Module):
    """mode : "uncoded" (sans paramètre) | "decoder" | "jscc"."""

    MODES = ("uncoded", "decoder", "jscc")

    def __init__(self, cfg, mode):
        super().__init__()
        assert mode in self.MODES
        self.cfg, self.mode = cfg, mode
        self.enc = Readout(cfg, bidir=False) if mode == "jscc" else None
        self.dec = Readout(cfg, bidir=True) if mode in ("decoder", "jscc") else None

    def encode(self, t, m):
        cfg = self.cfg
        g = gaps(t, m)
        if self.enc is None:
            return t, g, torch.zeros_like(g)
        x = torch.stack([g / cfg.mean_gap, t / cfg.Tc], -1) * m.unsqueeze(-1)
        a = cfg.a_max * torch.tanh(self.enc(x, g / cfg.mean_gap, m.sum(1)))
        gx = g * torch.exp(a) * m
        tot = gx.sum(1, keepdim=True)
        scale = (cfg.Tc / tot.clamp(min=1e-6)).clamp(max=1.0)   # X <= Tc, et = 1
        gx = gx * scale                                         # si deja dans la fenetre
        return torch.cumsum(gx, 1) * m, gx, a

    def forward(self, t, m, gen=None, sigma=None):
        cfg = self.cfg
        sigma = cfg.sigma_J if sigma is None else sigma
        X, gx, a = self.encode(t, m)
        Z = torch.randn(X.shape, generator=gen, device=X.device) * sigma
        Y = (X + Z) * m
        Ys = torch.sort(Y.masked_fill(~m, 1e6), 1).values * m   # le RX trie
        if self.dec is None:
            S_hat = Ys
        else:
            gy = gaps(Ys, m)
            x = torch.stack([gy / cfg.mean_gap, Ys / cfg.Tc], -1) * m.unsqueeze(-1)
            c_max = cfg.c_max_ratio * max(cfg.mean_gap, cfg.sigma_J)
            h = self.dec(x, gy.abs() / cfg.mean_gap, m.sum(1))
            S_hat = Ys + c_max * torch.tanh(h) * m
        return {"t": t, "m": m, "X": X, "gx": gx, "a": a, "Y": Y, "Ys": Ys,
                "S_hat": S_hat}

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def distortion(out):
    """Éq. 7-8 du PDF, exactement : (1/N) Σ (T_i - T̂_i)², appariée par indice,
    moyennée sur le batch. Pas de troncature min(N_S, N_Ŝ) : les deux comptes
    sont égaux par construction."""
    m = out["m"].to(out["t"].dtype)
    sq = (out["t"] - out["S_hat"]) ** 2 * m
    return (sq.sum(1) / m.sum(1).clamp(min=1)).mean()

### Vérification structurelle

Deux propriétés qui doivent être vraies **avant tout entraînement**. Si l'une casse
après une modification du code, c'est un bug, pas un résultat.

In [ ]:
_t, _m = SOURCES["hawkes"](CFG, 512, 0)
_g = torch.Generator().manual_seed(7)
_ds = []
for mode in MODES:
    torch.manual_seed(0)
    mdl = ResidualSystem(CFG, mode)
    with torch.no_grad():
        out = mdl(_t, _m, gen=torch.Generator().manual_seed(7))
        _ds.append(float(distortion(out)))
    n_s = _m.sum(1)
    n_x = (out["X"] > 0).sum(1) if mdl.enc is not None else n_s
    n_h = out["m"].sum(1)
    ok_order = bool((out["X"][:, 1:] - out["X"][:, :-1] >= 0)[_m[:, 1:]].all())
    ok_win = bool((out["X"] <= CFG.Tc + 1e-5).all())
    print(f"{mode:8s}  D_init={_ds[-1]:.6f}  n_params={mdl.n_params:4d}  "
          f"N_X=N_S : {bool((n_x == n_s).all())}   N_Ŝ=N_S : {bool((n_h == n_s).all())}   "
          f"X croissant : {ok_order}   X ≤ Tc : {ok_win}")

assert max(_ds) - min(_ds) < 1e-5, "les trois modes doivent coïncider à l'init"
print()
print(f"Les trois modes coïncident à l'init (écart {max(_ds)-min(_ds):.2e}) : OK.")
print(f"sigma_J^2 = {CFG.sigma_J**2:.6f} ; D_uncoded = {_ds[0]:.6f} "
      f"(< sigma^2 grâce au tri de Y au récepteur).")

## 6. Entraînement et mesure

In [ ]:
def train(cfg, mode, ds_train, steps=None, seed=0, log=None):
    steps = steps or cfg.steps
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 1)
    model = ResidualSystem(cfg, mode)
    hist = []
    if model.n_params:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        for st in range(1, steps + 1):
            t, m = ds_train.sample(cfg.batch, gen)
            loss = distortion(model(t, m, gen=gen))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()
            if log and (st % log == 0 or st == 1):
                hist.append((st, float(loss.detach())))
    return model, hist


@torch.no_grad()
def evaluate(model, cfg, ds_eval, seed=9999, batch=4000):
    gen = torch.Generator().manual_seed(seed)
    t, m = ds_eval.sample(batch, gen)
    out = model(t, m, gen=gen)
    d = float(distortion(out))
    pair = m[:, 1:] & m[:, :-1]
    reorder = float((((out["Y"][:, 1:] - out["Y"][:, :-1]) < 0) & pair)
                    .any(1).float().mean())
    return {"d": d, "rmse_gaps": math.sqrt(d) / cfg.mean_gap,
            "reorder_frac_Y": reorder, "out": out}


def make_datasets(name, cfg, n_train=N_TRAIN, n_eval=N_EVAL):
    t, m = SOURCES[name](cfg, n_train, 0)
    te, me = SOURCES[name](cfg, n_eval, 1)
    tab = blind_table(t[:8000], m[:8000])
    return Dataset(t, m), Dataset(te, me), blind_D(tab, te, me)

## 7. Expérience 1 — balayage du bruit, trois sources, trois systèmes

La comparaison contrôlée du PDF : même rate $R = 1$, même nombre d'événements
transmis $N_X = N_S$, même canal. Tout écart entre les trois systèmes vient donc
uniquement de la géométrie temporelle.

Chaque point est la moyenne sur `SEEDS` graines. Les deux références sont tracées :
$D_{\text{blind}}$ (plafond : ne rien transmettre) et $\sigma_J^2$ (transmission
brute sans même trier $Y$).

In [ ]:
def run_sigma_sweep(sources=("poisson", "gamma_k4", "hawkes"),
                    sigma_grid=SIGMA_GRID, seeds=SEEDS, steps=STEPS, verbose=True):
    rows, models = [], {}
    for src in sources:
        ds, dse, Db = make_datasets(src, CFG)
        for ratio in sigma_grid:
            cfg = replace(CFG, sigma_ratio=ratio, steps=steps)
            for mode in MODES:
                ds_, t0 = [], time.time()
                for sd in seeds:
                    mdl, _ = train(cfg, mode, ds, seed=sd)
                    ds_.append(evaluate(mdl, cfg, dse)["d"])
                    models[(src, ratio, mode, sd)] = (mdl, cfg)
                r = {"source": src, "sigma_ratio": ratio, "mode": mode,
                     "d_mse": float(np.mean(ds_)), "d_std": float(np.std(ds_)),
                     "rmse_gaps": math.sqrt(np.mean(ds_)) / CFG.mean_gap,
                     "d_over_blind": float(np.mean(ds_)) / Db,
                     "blind": Db, "sigma2": cfg.sigma_J ** 2}
                rows.append(r)
                if verbose:
                    print(f"{src:9s} σ/µ={ratio:4.1f} {mode:8s} "
                          f"D={r['d_mse']:.5f} ±{r['d_std']:.5f}  "
                          f"D/blind={r['d_over_blind']:.3f}  "
                          f"({time.time()-t0:.0f}s)", flush=True)
    return rows, models


SWEEP_ROWS, SWEEP_MODELS = run_sigma_sweep()

In [ ]:
show_table(SWEEP_ROWS,
           cols=["source", "sigma_ratio", "mode", "d_mse", "d_std", "rmse_gaps",
                 "d_over_blind", "blind", "sigma2"],
           title="Balayage du bruit — D en unités de Ts², rmse_gaps en écarts moyens")
print()
for src in dict.fromkeys(r["source"] for r in SWEEP_ROWS):
    for ratio in SIGMA_GRID:
        sel = {r["mode"]: r["d_mse"] for r in SWEEP_ROWS
               if r["source"] == src and r["sigma_ratio"] == ratio}
        print(f"{src:9s} σ/µ={ratio:4.1f}  gain décodeur = {sel['uncoded']/sel['decoder']:5.2f}×"
              f"   gain encodeur (jscc/decoder) = {sel['decoder']/sel['jscc']:5.3f}×")

In [ ]:
def fig_sigma_sweep(rows, sources=("poisson", "gamma_k4", "hawkes")):
    fig, axes = plt.subplots(1, len(sources), figsize=(4.6 * len(sources), 4.4),
                             facecolor=SURFACE, sharey=True)
    for ax, src in zip(np.atleast_1d(axes), sources):
        sub = [r for r in rows if r["source"] == src]
        for mode in MODES:
            pts = sorted([r for r in sub if r["mode"] == mode],
                         key=lambda r: r["sigma_ratio"])
            colour, marker = MODE_STYLE[mode]
            xs = [r["sigma_ratio"] for r in pts]
            ys = [r["d_mse"] for r in pts]
            es = [r["d_std"] for r in pts]
            ax.errorbar(xs, ys, yerr=es, color=colour, lw=2.0, marker=marker, ms=7,
                        mew=2.0, mfc=SURFACE, mec=colour, capsize=3, zorder=5,
                        label=mode)
        xs = sorted({r["sigma_ratio"] for r in sub})
        ax.axhline(sub[0]["blind"], color=C_BLIND, lw=1.5, ls=(0, (4, 3)), zorder=4,
                   label="aveugle (rien transmis)")
        ax.plot(xs, [(x * CFG.mean_gap) ** 2 for x in xs], color=C_OPT, lw=1.2,
                ls=(0, (1, 2)), zorder=4, label="$\\sigma_J^2$ (sans tri)")
        ax.set_xscale("log"); ax.set_yscale("log")
        style(ax, "$\\sigma_J$ / écart moyen  (log)",
              "D  (unités de $T_s^2$)" if src == sources[0] else "", src)
    legend(np.atleast_1d(axes)[-1], loc="lower right")
    fig.suptitle("Distorsion vs bruit — tout ce qui est sous la ligne grise transmet "
                 "réellement de l'information", color=INK, fontsize=11, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


fig_sigma_sweep(SWEEP_ROWS); plt.show()

**Lecture.** Les trois courbes apprises restent nettement **sous** la ligne grise :
contrairement à la v1, de l'information temporelle traverse effectivement le
système, à tous les niveaux de bruit. `decoder` bat `uncoded` partout (c'est
garanti par l'initialisation), et `jscc` bat `decoder` d'une marge faible mais
systématique — c'est le gain d'encodeur, à isoler et à interpréter dans les
expériences suivantes.

## 7bis. Sanity check — sans bruit, reconstruit-on bien ?

Trois contrôles, du plus structurel au plus informatif. Les deux premiers portent
un `assert` : s'ils sautent, c'est un bug, pas un résultat.

- **A.** Modèle **non entraîné**, $\sigma_J = 0$. Le canal est l'identité, les têtes
  sont à zéro, donc $X = S$, $Y = X$, $\hat S = Y$ : la reconstruction doit être
  exacte à la précision machine, pour les trois modes.
- **B.** Modèle **entraîné à $\sigma_J = 0$**. Comme $D = 0$ dès l'initialisation, le
  gradient est nul et l'entraînement ne doit rien casser. C'est un contrôle de
  non-régression du chemin de gradient.
- **C.** Modèle **entraîné à $\sigma_J = \mu$, réévalué à $\sigma_J = 0$**. Celui-là est
  le plus parlant, et $D$ n'y retombe **pas** à zéro : c'est attendu et il faut savoir
  le dire. Le décodeur a appris une correction calibrée sur un niveau de bruit
  donné ; appliquée sans bruit, cette correction devient un biais. Le couple
  encodeur/décodeur est **accordé à un $\sigma_J$**, ce n'est pas un codec inversible.

In [ ]:
ds_p, dse_p, _ = make_datasets("poisson", CFG, n_train=8000, n_eval=2000)
cfg_zero = replace(CFG, sigma_ratio=0.0)
_t, _m = SOURCES["poisson"](cfg_zero, 2000, 4242)

print("A. Modèle NON entraîné, sigma_J = 0")
for mode in MODES:
    torch.manual_seed(0)
    mdl = ResidualSystem(cfg_zero, mode)
    with torch.no_grad():
        o = mdl(_t, _m, gen=torch.Generator().manual_seed(1))
    err = float(((o["t"] - o["S_hat"]).abs() * o["m"]).max())
    n_ok = bool((o["m"].sum(1) == _m.sum(1)).all())
    print(f"   {mode:8s}  D = {float(distortion(o)):.3e}   "
          f"erreur max = {err:.2e}  ({err/CFG.mean_gap:.1e} écart moyen)   "
          f"N conservé : {n_ok}")
    assert err < 1e-6, "sans bruit, la reconstruction doit être exacte"

print()
print("B. Système ENTRAÎNÉ à sigma_J = 0 : doit rester l'identité")
cfg_zt = replace(CFG, sigma_ratio=0.0, steps=200)
for mode in ("decoder", "jscc"):
    mdl, _ = train(cfg_zt, mode, ds_p, seed=0)
    with torch.no_grad():
        o = mdl(_t, _m, gen=torch.Generator().manual_seed(1))
    d0 = float(distortion(o))
    print(f"   {mode:8s}  D après 200 pas = {d0:.3e}")
    assert d0 < 1e-6, "l'entraînement à bruit nul ne doit pas dégrader l'identité"

print()
print("C. Système entraîné à sigma_J = µ, réévalué à sigma_J = 0")
for mode in ("decoder", "jscc"):
    mdl, cfg_ = SWEEP_MODELS[("poisson", 1.0, mode, SEEDS[0])]
    with torch.no_grad():
        o_n = mdl(_t, _m, gen=torch.Generator().manual_seed(1))
        o_0 = mdl(_t, _m, gen=torch.Generator().manual_seed(1), sigma=0.0)
    extra = ""
    if mode == "jscc":
        dev = float(((o_0["X"] - o_0["t"]).abs() * o_0["m"]).sum() / o_0["m"].sum())
        extra = f"      déformation |X - S| = {dev/CFG.mean_gap:.2f} écart moyen"
    print(f"   {mode:8s}  D à sigma_J = µ : {float(distortion(o_n)):.5f}"
          f"   ->   D à sigma_J = 0 : {float(distortion(o_0)):.5f}{extra}")
print()
print("   D ne retombe pas à zéro, et c'est normal : la correction du décodeur est")
print("   calibrée pour un niveau de bruit ; sans bruit elle devient un biais.")
print("   Le couple encodeur/décodeur est accordé à un sigma_J donné.")

### Chaîne complète sur une source de Poisson — `uncoded` contre `jscc`

Même tirage, même graine de bruit pour les deux systèmes : comme $N_X = N_S$ des
deux côtés, le vecteur $Z$ s'applique élément par élément à l'identique, donc la
comparaison est directe. Les traits fins relient chaque $T_i$ à son $\hat T_i$ :
leur inclinaison est l'erreur.

Le troisième panneau rejoue le **même** modèle `jscc`, mêmes poids, avec
$\sigma_J = 0$ — c'est la version visuelle du contrôle C ci-dessus.

In [ ]:
def fig_chain_poisson(models, ratio=1.0, block=0, seed=11):
    mdl_u, cfg = models[("poisson", ratio, "uncoded", SEEDS[0])]
    mdl_j, _ = models[("poisson", ratio, "jscc", SEEDS[0])]
    t, m = SOURCES["poisson"](cfg, block + 1, seed)
    with torch.no_grad():
        ou = mdl_u(t, m, gen=torch.Generator().manual_seed(seed))
        oj = mdl_j(t, m, gen=torch.Generator().manual_seed(seed))
        oz = mdl_j(t, m, gen=torch.Generator().manual_seed(seed), sigma=0.0)
    pick = lambda a: a[block][m[block]].numpy()
    dblock = lambda o: float(((o["t"] - o["S_hat"]) ** 2 * o["m"])[block].sum()
                             / m[block].sum())
    panels = [("uncoded", ou, f"$X = S$, $\\hat S = Y$ trié — D = {dblock(ou):.5f}"),
              ("jscc", oj, f"encodeur + décodeur appris — D = {dblock(oj):.5f}"),
              ("jscc, $\\sigma_J = 0$", oz,
               f"mêmes poids, canal parfait — D = {dblock(oz):.5f}")]
    fig, axes = plt.subplots(len(panels), 1, figsize=(9.6, 3.6 * len(panels)),
                             facecolor=SURFACE, sharex=True)
    for ax, (name, o, sub) in zip(axes, panels):
        rows = [("$S$  source", pick(o["t"]), INK),
                ("$X$  mot de code", pick(o["X"]), C_UNCODED),
                ("$Y$  sortie canal", pick(o["Ys"]), C_DECODER),
                ("$\\hat S$  reconstruction", pick(o["S_hat"]), C_JSCC)]
        for k, (lab, vals, col) in enumerate(rows):
            y0 = len(rows) - 1 - k
            ax.eventplot([vals], colors=col, lineoffsets=y0, linelengths=0.52,
                         linewidths=2.4, zorder=6)
            ax.text(-0.015, y0, lab, transform=ax.get_yaxis_transform(), color=col,
                    fontsize=9, fontweight="bold", ha="right", va="center")
        for a_, b_ in zip(pick(o["t"]), pick(o["S_hat"])):
            ax.plot([a_, b_], [3, 0], color=MUTED, lw=0.7, alpha=0.55, zorder=3)
        for xv in (0.0, cfg.Tc):
            ax.axvline(xv, color=MUTED, lw=1.2, ls=(0, (4, 3)), zorder=2)
        ax.set_ylim(-0.6, 3.6); ax.set_yticks([])
        style(ax, "temps  (unités de $T_s$)", "", name, sub)
    n = int(m[block].sum())
    fig.suptitle(f"Source Poisson, bloc {block} — "
                 f"$N_S = N_X = N_{{\\hat S}} = {n}$, $\\sigma_J/\\mu$ = {ratio:g}, "
                 f"même tirage et même bruit", color=INK, fontsize=11, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    return fig


for b in range(2):
    fig_chain_poisson(SWEEP_MODELS, block=b); plt.show()

## 8. Expérience 2 — Step 2 du PDF : la prévisibilité crée-t-elle le gain ?

Question centrale du Step 2 : $W_i \sim \Gamma(k, \mu/k)$ à moyenne fixée, $k$
balayé dans $\{1, 2, 4, 8\}$. $k = 1$ est Poisson, $k$ grand donne un timing de plus
en plus régulier. Prédiction : une source plus prévisible laisse plus de marge au
décodeur (il peut s'appuyer sur l'a priori) **et** à l'encodeur (la distribution des
écarts est plus concentrée, donc plus facile à redéployer).

On sépare les deux effets en lisant `uncoded/decoder` (gain décodeur) et
`decoder/jscc` (gain encodeur) séparément.

In [ ]:
K_GRID = [("poisson", 1), ("gamma_k2", 2), ("gamma_k4", 4), ("gamma_k8", 8)]
K_SIGMA = 1.0

K_ROWS, K_MODELS = [], {}
for name, k in K_GRID:
    ds, dse, Db = make_datasets(name, CFG)
    cfg = replace(CFG, sigma_ratio=K_SIGMA, steps=STEPS)
    res = {}
    for mode in MODES:
        vals = []
        for sd in SEEDS:
            mdl, _ = train(cfg, mode, ds, seed=sd)
            vals.append(evaluate(mdl, cfg, dse)["d"])
            K_MODELS[(k, mode, sd)] = (mdl, cfg)
        res[mode] = float(np.mean(vals))
    K_ROWS.append({"k": k, "blind": Db, **res,
                   "gain_decodeur": res["uncoded"] / res["decoder"],
                   "gain_encodeur": res["decoder"] / res["jscc"],
                   "gain_total": res["uncoded"] / res["jscc"]})
    print(f"k={k}  uncoded={res['uncoded']:.5f} decoder={res['decoder']:.5f} "
          f"jscc={res['jscc']:.5f}  blind={Db:.5f}  "
          f"gain_dec={K_ROWS[-1]['gain_decodeur']:.2f}× "
          f"gain_enc={K_ROWS[-1]['gain_encodeur']:.3f}×", flush=True)

show_table(K_ROWS, cols=["k", "uncoded", "decoder", "jscc", "blind",
                         "gain_decodeur", "gain_encodeur", "gain_total"],
           title=f"Step 2 — renouvellement Gamma, σ_J/µ = {K_SIGMA}")

In [ ]:
def fig_k_sweep(rows):
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4), facecolor=SURFACE)
    ks = [r["k"] for r in rows]
    for mode in MODES:
        colour, marker = MODE_STYLE[mode]
        axes[0].plot(ks, [r[mode] for r in rows], color=colour, lw=2.0, marker=marker,
                     ms=8, mew=2.0, mfc=SURFACE, mec=colour, zorder=5, label=mode)
    axes[0].plot(ks, [r["blind"] for r in rows], color=C_BLIND, lw=1.5,
                 ls=(0, (4, 3)), zorder=4, label="aveugle")
    axes[0].set_yscale("log")
    style(axes[0], "k  (forme de la Gamma)", "D  (unités de $T_s^2$)",
          "Distorsion vs régularité de la source",
          f"$\\sigma_J$ / écart moyen = {K_SIGMA}")
    legend(axes[0])

    axes[1].plot(ks, [r["gain_decodeur"] for r in rows], color=C_DECODER, lw=2.0,
                 marker="s", ms=8, mew=2.0, mfc=SURFACE, mec=C_DECODER,
                 label="gain décodeur  (uncoded / decoder)")
    axes[1].plot(ks, [r["gain_encodeur"] for r in rows], color=C_JSCC, lw=2.0,
                 marker="^", ms=8, mew=2.0, mfc=SURFACE, mec=C_JSCC,
                 label="gain encodeur  (decoder / jscc)")
    axes[1].axhline(1.0, color=INK, lw=1.2, ls=(0, (4, 3)), zorder=4)
    style(axes[1], "k  (forme de la Gamma)", "facteur de gain",
          "D'où vient le gain ?", "tirets : aucun gain")
    legend(axes[1])
    fig.tight_layout()
    return fig


fig_k_sweep(K_ROWS); plt.show()

## 9. Expérience 3 — qu'apprend l'encodeur ? (éq. 12)

On trace $g_i \mapsto g^{(X)}_i$ ainsi que le multiplicateur appris
$g^{(X)}_i / g_i$ en fonction de la taille de l'écart source. Comme $N_X = N_S$
est garanti et que les deux suites sont ordonnées, le $i$-ème écart du code
correspond bien au $i$-ème écart de la source.

**Hypothèse à tester.** Le budget est $\sum_i g^X_i \le T_c$ et la source remplit
déjà la fenêtre, donc l'encodeur ne peut pas dilater globalement : il ne peut que
**réallouer**. Or le jitter $\sigma_J$ est absolu, il détruit surtout les petits
écarts, alors que les longs silences restent estimables (et le décodeur peut
s'appuyer sur l'a priori pour eux). On attend donc un **compandage** : multiplicateur
supérieur à 1 sur les petits écarts, inférieur à 1 sur les grands.

In [ ]:
def fig_gap_mapping(models, src="gamma_k4", ratios=SIGMA_GRID, seed=4242, n_blocks=1500):
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), facecolor=SURFACE)
    colours = plt.get_cmap("viridis")(np.linspace(0.15, 0.8, len(ratios)))
    lim = 0.0
    for colour, ratio in zip(colours, ratios):
        mdl, cfg = models[(src, ratio, "jscc", SEEDS[0])]
        t, m = SOURCES[src](cfg, n_blocks, seed)
        with torch.no_grad():
            _, gx, _ = mdl.encode(t, m)
        g = gaps(t, m)
        sel = m.numpy().ravel()
        gs = (g.numpy().ravel() / cfg.mean_gap)[sel]
        gxs = (gx.numpy().ravel() / cfg.mean_gap)[sel]
        lim = max(lim, np.percentile(gs, 99.5), np.percentile(gxs, 99.5))
        axes[0].scatter(gs, gxs, s=4, color=colour, alpha=0.25, lw=0,
                        label=f"$\\sigma_J/\\mu$ = {ratio:g}")
        # médiane glissante du multiplicateur
        order = np.argsort(gs)
        gs_o, mult_o = gs[order], (gxs / np.clip(gs, 1e-6, None))[order]
        w = max(len(gs_o) // 60, 20)
        xs = [gs_o[i:i + w].mean() for i in range(0, len(gs_o) - w, w)]
        ys = [np.median(mult_o[i:i + w]) for i in range(0, len(gs_o) - w, w)]
        axes[1].plot(xs, ys, color=colour, lw=2.0, label=f"$\\sigma_J/\\mu$ = {ratio:g}")

    axes[0].plot([0, lim], [0, lim], color=INK, lw=1.5, ls=(0, (4, 3)), zorder=6)
    axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
    style(axes[0], "$g_i$ / écart moyen", "$g_i^{(X)}$ / écart moyen",
          "Application des écarts (éq. 12)", f"source = {src} ; tirets : identité")
    legend(axes[0])

    axes[1].axhline(1.0, color=INK, lw=1.5, ls=(0, (4, 3)), zorder=4)
    axes[1].set_xscale("log")
    style(axes[1], "$g_i$ / écart moyen  (log)", "multiplicateur médian  $g^{(X)}_i/g_i$",
          "Compandage : les petits écarts sont-ils dilatés ?",
          "au-dessus de 1 : dilaté ; en dessous : comprimé")
    legend(axes[1])
    fig.tight_layout()
    return fig


fig_gap_mapping(SWEEP_MODELS); plt.show()

In [ ]:
# Chiffrage direct : multiplicateur médian par décile d'écart source
mdl, cfg = SWEEP_MODELS[("gamma_k4", SIGMA_GRID[-1], "jscc", SEEDS[0])]
t, m = SOURCES["gamma_k4"](cfg, 3000, 77)
with torch.no_grad():
    _, gx, _ = mdl.encode(t, m)
g = gaps(t, m)
sel = m.numpy().ravel()
gs = (g.numpy().ravel() / cfg.mean_gap)[sel]
mult = (gx.numpy().ravel()[sel]) / np.clip(g.numpy().ravel()[sel], 1e-9, None)
qs = np.quantile(gs, np.linspace(0, 1, 11))
print(f"source = gamma_k4, σ_J/µ = {cfg.sigma_ratio}")
print(" décile d'écart source      multiplicateur médian appris")
for i in range(10):
    s = (gs >= qs[i]) & (gs < qs[i + 1])
    if s.sum():
        print(f"  [{qs[i]:5.2f}, {qs[i+1]:5.2f}] µ  ->  {np.median(mult[s]):.3f}")
print(f"\nSomme des écarts source / Tc      = {float((g.sum(1)/cfg.Tc).mean()):.3f}")
print(f"Somme des écarts encodés / Tc     = {float((gx.sum(1)/cfg.Tc).mean()):.3f}")
print("La fenêtre est déjà pleine : l'encodeur ne peut que réallouer, pas dilater.")

## 10. Expérience 4 — Step 3 du PDF : le modèle minimal, avec son optimum

C'est la partie qui répond réellement à la question de recherche. On réduit à
$S = \{T_1, T_2\}$, soit la variable unique $G = T_2 - T_1$, avec

$$G \longrightarrow G_X \longrightarrow G_Y = G_X + Z \longrightarrow \hat G .$$

Contrainte : $G_X \in [0, T_c]$ — la version deux-événements de la contrainte
*sample-wise* $\sum_i g^X_i \le T_c$ du problème complet.

**Ce qui change tout par rapport à une simple répétition en petit : on calcule
l'optimum.** Sur une grille, l'encodeur est une fonction monotone libre et le
décodeur MMSE associé est $\hat G(y) = \mathbb{E}[G \mid G_Y = y]$, calculable
exactement par intégration numérique. On optimise directement la table de
l'encodeur par descente de gradient. On obtient trois références exactes :

- `uncoded` : $\sigma^2$ ;
- `decoder-opt` : identité à l'émission + MMSE exact à la réception ;
- `jscc-opt` : encodeur monotone optimal + MMSE exact.

Le réseau est ensuite comparé à **ces** chiffres, pas à `uncoded`.

In [ ]:
def source_pdf(kind, g, mu, k=4.0):
    if kind == "exp":
        p = np.exp(-g / mu) / mu
    elif kind == "gamma":
        p = g ** (k - 1) * np.exp(-g * k / mu) * (k / mu) ** k / math.gamma(k)
    elif kind == "bimodal":
        p = (0.5 * np.exp(-(g - 0.4 * mu) ** 2 / (2 * (0.12 * mu) ** 2))
             + 0.5 * np.exp(-(g - 2.2 * mu) ** 2 / (2 * (0.35 * mu) ** 2)))
    else:
        raise ValueError(kind)
    p = np.clip(p, 0, None)
    return p / p.sum()


class Grid:
    """Discrétisation commune source / canal, pour le calcul exact de la MMSE."""

    def __init__(self, kind, mu, Tc, sigma, M=161, My=321, k=4.0):
        self.g = torch.tensor(np.linspace(1e-4, Tc, M), dtype=torch.float64)
        self.p = torch.tensor(source_pdf(kind, self.g.numpy(), mu, k), dtype=torch.float64)
        self.y = torch.tensor(np.linspace(-4 * sigma, Tc + 4 * sigma, My),
                              dtype=torch.float64)
        self.dy = float(self.y[1] - self.y[0])
        self.sigma, self.Tc, self.mu, self.kind = sigma, Tc, mu, kind

    def distortion(self, x):
        """x (M,) : images de la grille par l'encodeur -> (D, Ĝ(y), p(y))."""
        d = self.y.view(1, -1) - x.view(-1, 1)
        q = self.p.view(-1, 1) * torch.exp(-d ** 2 / (2 * self.sigma ** 2))
        marg = q.sum(0) + 1e-300
        ghat = (q * self.g.view(-1, 1)).sum(0) / marg
        err = (self.g.view(-1, 1) - ghat.view(1, -1)) ** 2
        D = (q * err).sum() * self.dy / (math.sqrt(2 * math.pi) * self.sigma)
        return D, ghat, marg


def monotone_map(w, Tc):
    """Paramètre libre -> fonction strictement croissante à valeurs dans (0, Tc]."""
    return Tc * torch.cumsum(torch.softmax(w, 0), 0)


def optimal_encoder(grid, steps=1500, lr=0.05, seed=0):
    torch.manual_seed(seed)
    M = grid.g.shape[0]
    w = nn.Parameter(torch.log(torch.full((M,), 1.0 / M, dtype=torch.float64)))
    opt = torch.optim.Adam([w], lr=lr)
    for _ in range(steps):
        D, _, _ = grid.distortion(monotone_map(w, grid.Tc))
        opt.zero_grad(); D.backward(); opt.step()
    with torch.no_grad():
        x = monotone_map(w, grid.Tc)
        D, _, _ = grid.distortion(x)
    return x.detach(), float(D)


def reference_levels(grid):
    """(D_uncoded, D_decoder-opt) : identité à l'émission, sans / avec MMSE."""
    D_dec, _, _ = grid.distortion(grid.g.clone())
    return grid.sigma ** 2, float(D_dec)

In [ ]:
class TinyEnc(nn.Module):
    """Encodeur scalaire appris, initialisé à l'identité (dernière couche à zéro)."""

    def __init__(self, Tc, mu, h=64):
        super().__init__()
        self.Tc, self.mu = Tc, mu
        self.net = nn.Sequential(nn.Linear(1, h), nn.Tanh(), nn.Linear(h, h),
                                 nn.Tanh(), nn.Linear(h, 1))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)

    def forward(self, g):
        a = torch.tanh(self.net((g / self.mu).unsqueeze(-1)).squeeze(-1))
        return (g * torch.exp(math.log(6.0) * a)).clamp(0.0, self.Tc)


class TinyDec(nn.Module):
    """Décodeur scalaire appris, initialisé à l'identité."""

    def __init__(self, Tc, mu, cmax, h=64):
        super().__init__()
        self.Tc, self.mu, self.cmax = Tc, mu, cmax
        self.net = nn.Sequential(nn.Linear(1, h), nn.Tanh(), nn.Linear(h, h),
                                 nn.Tanh(), nn.Linear(h, 1))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)

    def forward(self, y):
        return y + self.cmax * torch.tanh(self.net((y / self.mu).unsqueeze(-1)).squeeze(-1))


def sample_G(grid, n, gen):
    idx = torch.multinomial(grid.p.float(), n, replacement=True, generator=gen)
    return grid.g[idx].float()


def train_tiny(grid, mode, steps=2000, batch=4096, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 1)
    mu, Tc, sg = grid.mu, grid.Tc, grid.sigma
    enc = TinyEnc(Tc, mu) if mode == "jscc" else None
    dec = TinyDec(Tc, mu, 3 * max(mu, sg)) if mode in ("decoder", "jscc") else None
    params = [p for m in (enc, dec) if m is not None for p in m.parameters()]
    if params:
        opt = torch.optim.Adam(params, lr=lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        for _ in range(steps):
            g = sample_G(grid, batch, gen)
            x = g if enc is None else enc(g)
            y = x + torch.randn(x.shape, generator=gen) * sg
            gh = y if dec is None else dec(y)
            loss = ((g - gh) ** 2).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sch.step()
    with torch.no_grad():
        g = sample_G(grid, 200_000, torch.Generator().manual_seed(123))
        x = g if enc is None else enc(g)
        y = x + torch.randn(x.shape, generator=torch.Generator().manual_seed(124)) * sg
        gh = y if dec is None else dec(y)
        D = float(((g - gh) ** 2).mean())
    return enc, dec, D

In [ ]:
STEP3_KINDS = ["exp", "gamma", "bimodal"]
STEP3_SIGMAS = [0.3, 1.0, 3.0]
MU, TC = CFG.mean_gap, CFG.Tc

STEP3_ROWS, STEP3_GRIDS = [], {}
for kind in STEP3_KINDS:
    for sr in STEP3_SIGMAS:
        G = Grid(kind, MU, TC, sr * MU)
        D_unc, D_dec_opt = reference_levels(G)
        x_opt, D_jscc_opt = optimal_encoder(G)
        learned, enc_jscc = {}, None
        for md_ in MODES:
            e_, _, d_ = train_tiny(G, md_)
            learned[md_] = d_
            if md_ == "jscc":
                enc_jscc = e_
        STEP3_GRIDS[(kind, sr)] = (G, x_opt, enc_jscc)
        STEP3_ROWS.append({
            "source": kind, "sigma_ratio": sr,
            "opt_uncoded": D_unc, "opt_decoder": D_dec_opt, "opt_jscc": D_jscc_opt,
            "appris_uncoded": learned["uncoded"], "appris_decoder": learned["decoder"],
            "appris_jscc": learned["jscc"],
            "gain_enc_opt": D_dec_opt / D_jscc_opt,
            "gain_enc_appris": learned["decoder"] / learned["jscc"],
            "ecart_a_opt": learned["jscc"] / D_jscc_opt})
        print(f"{kind:8s} σ/µ={sr:4.1f} | OPT unc={D_unc:.5f} dec={D_dec_opt:.5f} "
              f"jscc={D_jscc_opt:.5f} (gain enc {D_dec_opt/D_jscc_opt:.2f}×) | "
              f"APPRIS dec={learned['decoder']:.5f} jscc={learned['jscc']:.5f} "
              f"(à {learned['jscc']/D_jscc_opt:.2f}× de l'optimum)", flush=True)

show_table(STEP3_ROWS, title="Step 3 — modèle à deux événements, optimum numérique vs appris")

In [ ]:
def fig_step3(kind="gamma", sr=1.0):
    G, x_opt, enc = STEP3_GRIDS[(kind, sr)]
    g = G.g.numpy() / MU
    xo = x_opt.numpy() / MU
    with torch.no_grad():
        xl = enc(G.g.float()).numpy() / MU
    p = G.p.numpy()

    fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.4), facecolor=SURFACE)

    ax = axes[0]
    ax.plot(g, g, color=INK, lw=1.5, ls=(0, (4, 3)), label="identité (uncoded)")
    ax.plot(g, xo, color=C_OPT, lw=2.4, label="encodeur optimal")
    ax.plot(g, xl, color=C_JSCC, lw=2.0, ls="--", label="encodeur appris")
    ax.set_ylim(0, TC / MU)
    style(ax, "$G$ / écart moyen", "$G_X$ / écart moyen",
          "L'encodeur optimal dilate", f"{kind}, $\\sigma_J/\\mu$ = {sr:g}")
    legend(ax, "upper left")

    ax = axes[1]
    ax.fill_between(g, 0, p / p.max(), color=C_BLIND, alpha=0.35, lw=0,
                    label="densité de $G$")
    slope_o = np.gradient(xo, g)
    slope_l = np.gradient(xl, g)
    ax2 = ax.twinx()
    ax2.plot(g, slope_o, color=C_OPT, lw=2.4, label="pente optimale $dG_X/dG$")
    ax2.plot(g, slope_l, color=C_JSCC, lw=2.0, ls="--", label="pente apprise")
    ax2.axhline(1.0, color=INK, lw=1.2, ls=(0, (4, 3)))
    ax2.set_ylabel("pente", color=INK2, fontsize=10)
    ax2.tick_params(colors=INK2, labelsize=9)
    ax.set_xlim(0, np.percentile(g[p > p.max() * 1e-3], 99))
    style(ax, "$G$ / écart moyen", "densité (normalisée)",
          "Le gain vient de la pente", "pente > 1 : SNR effectif amélioré")
    legend(ax2, "upper right")
    legend(ax, "upper left")

    ax = axes[2]
    rows = [r for r in STEP3_ROWS if r["source"] == kind]
    xs = [r["sigma_ratio"] for r in rows]
    ax.plot(xs, [r["opt_uncoded"] for r in rows], color=C_UNCODED, lw=2.0, marker="o",
            ms=7, mfc=SURFACE, mew=2, mec=C_UNCODED, label="uncoded")
    ax.plot(xs, [r["opt_decoder"] for r in rows], color=C_DECODER, lw=2.0, marker="s",
            ms=7, mfc=SURFACE, mew=2, mec=C_DECODER, label="décodeur optimal")
    ax.plot(xs, [r["opt_jscc"] for r in rows], color=C_OPT, lw=2.4, marker="D",
            ms=7, mfc=SURFACE, mew=2, mec=C_OPT, label="JSCC optimal")
    ax.plot(xs, [r["appris_jscc"] for r in rows], color=C_JSCC, lw=2.0, ls="--",
            marker="^", ms=7, mfc=SURFACE, mew=2, mec=C_JSCC, label="JSCC appris")
    ax.set_xscale("log"); ax.set_yscale("log")
    style(ax, "$\\sigma_J$ / écart moyen  (log)", "D  (unités de $T_s^2$)",
          "Le réseau atteint-il l'optimum ?", kind)
    legend(ax, "lower right")

    fig.tight_layout()
    return fig


fig_step3("gamma", 1.0); plt.show()
fig_step3("bimodal", 1.0); plt.show()

In [ ]:
# Chiffrage de la dilatation : où va la masse de probabilité après encodage ?
for kind in STEP3_KINDS:
    G, x_opt, _ = STEP3_GRIDS[(kind, 1.0)]
    g, x, p = G.g.numpy(), x_opt.numpy(), G.p.numpy()
    cum = np.cumsum(p)
    print(f"\n{kind} :")
    for q in (0.1, 0.5, 0.9):
        j = int(np.searchsorted(cum, q))
        print(f"  quantile {q:.0%} : G = {g[j]/MU:5.2f} µ  ->  G_X = {x[j]/MU:5.2f} µ"
              f"   (facteur {x[j]/g[j]:5.2f})")
    j50 = int(np.searchsorted(cum, 0.5))
    print(f"  pente locale à la médiane : dG_X/dG = {np.gradient(x, g)[j50]:.2f}"
          f"   ->  SNR effectif × {np.gradient(x, g)[j50]**2:.1f}")

**Le mécanisme, en une phrase.** Dans le modèle à deux événements, $G \sim \mu = T_c/9$
mais la fenêtre va jusqu'à $T_c$ : elle est essentiellement vide. L'encodeur optimal
**étale** la distribution de $G$ sur toute la fenêtre, avec une pente locale de
l'ordre de 6 là où se trouve la masse de probabilité. Le jitter $\sigma_J$ étant
absolu, multiplier les écarts par $\lambda$ multiplie le SNR effectif du canal de
timing par $\lambda^2$ — **à coût d'événements strictement constant**. C'est
exactement le type de gain que la question de recherche du PDF vise : l'information
est portée par la géométrie temporelle relative, pas par les temps individuels.

Le décodeur appris atteint le MMSE exact (écart < 3 %). L'encodeur appris capture la
bonne forme mais reste à un facteur 1.3 à 2.6 de l'optimum : c'est l'écart qu'il
reste à expliquer, et c'est un vrai sujet de discussion, pas un bug.

**Pourquoi le gain est bien plus petit dans le problème complet.** Avec $N$
événements, $\sum_i g_i \approx T_c$ : la fenêtre est **déjà pleine** (vérifié dans
l'expérience 3). L'encodeur ne peut donc pas dilater globalement, seulement
réallouer entre écarts — d'où un gain de 10-20 % au lieu de 2-10×. Cette différence
entre Step 3 et le problème complet n'est pas une incohérence, c'est la conséquence
directe de la contrainte de fenêtre, et c'est probablement le résultat le plus
intéressant du notebook.

## 11. Une réalisation à travers la chaîne

In [ ]:
def fig_chain(models, src="gamma_k4", ratio=SIGMA_GRID[-1], block=0, seed=7):
    mdl_j, cfg = models[(src, ratio, "jscc", SEEDS[0])]
    mdl_u, _ = models[(src, ratio, "uncoded", SEEDS[0])]
    t, m = SOURCES[src](cfg, block + 1, seed)
    with torch.no_grad():
        oj = mdl_j(t, m, gen=torch.Generator().manual_seed(seed))
        ou = mdl_u(t, m, gen=torch.Generator().manual_seed(seed))
    pick = lambda a: a[block][m[block]].numpy()
    rows = [("$S$  source", pick(oj["t"]), INK),
            ("$X$  mot de code", pick(oj["X"]), C_UNCODED),
            ("$Y$  sortie canal", pick(oj["Ys"]), C_DECODER),
            ("$\\hat S$  jscc", pick(oj["S_hat"]), C_JSCC),
            ("$\\hat S$  uncoded", pick(ou["S_hat"]), C_OPT)]
    fig, ax = plt.subplots(figsize=(9.4, 4.8), facecolor=SURFACE)
    for k, (name, vals, colour) in enumerate(rows):
        y0 = len(rows) - 1 - k
        ax.eventplot([vals], colors=colour, lineoffsets=y0, linelengths=0.52,
                     linewidths=2.4, zorder=6)
        ax.text(-0.015, y0, name, transform=ax.get_yaxis_transform(), color=colour,
                fontsize=9, fontweight="bold", ha="right", va="center")
    for xv in (0.0, cfg.Tc):
        ax.axvline(xv, color=MUTED, lw=1.2, ls=(0, (4, 3)), zorder=2)
    n = int(m[block].sum())
    ax.set_ylim(-0.6, len(rows) - 0.4); ax.set_yticks([])
    style(ax, "temps  (unités de $T_s$)", "", "Chaîne complète — même tirage, même bruit",
          f"source = {src}, $\\sigma_J/\\mu$ = {ratio:g}, "
          f"N_S = N_X = N_Ŝ = {n} (garanti par construction), tirets : $[0, T_c]$")
    fig.tight_layout()
    return fig


for b in range(3):
    fig_chain(SWEEP_MODELS, block=b); plt.show()

## 12. Expérience 5 — l'encodeur doit-il, lui aussi, voir tout le bloc ?

Le décodeur est bidirectionnel : il voit le bloc entier avant de produire $\hat S$.
L'encodeur, lui, est **causal** — $X_i$ ne dépend que de $g_1,\dots,g_i$ et $T_i$,
jamais des événements futurs du même bloc.

Rien dans l'énoncé n'impose cette asymétrie : la formulation par bloc
(*« for a source point-process block $S$ observed over duration $T_s$, the
encoder produces $X$ »*) autorise tout aussi bien un encodeur qui voit le bloc
entier avant d'émettre le premier événement codé — exactement l'argument qui
justifie déjà le décodeur bidirectionnel.

**Pourquoi ça pourrait aider.** La contrainte de fenêtre est **globale** :

$$\lambda = \min\!\left(1,\ \frac{T_c}{\sum_{j=1}^{N} \tilde g^X_j}\right)$$

Cette somme n'est connue qu'à la fin du bloc. Un encodeur causal doit décider
$g^X_i$ sans savoir combien d'événements vont encore arriver ni quelle sera leur
taille : il fait des paris locaux, que $\lambda$ peut ensuite écraser
uniformément si le pari était trop optimiste. Un encodeur qui voit tout le bloc
peut résoudre une allocation globale plutôt qu'une suite de décisions myopes —
plus proche de ce que fait l'encodeur optimal du Step 3 sur sa fenêtre.

**Ce qu'il en coûte.** Deux passes au lieu d'une : l'encodeur passe de 193 à
385 paramètres et de $N$ à $2N$ itérations LIF, et surtout perd toute latence
utile — il doit attendre la fin du bloc avant d'émettre le premier spike codé.
C'est le prix à payer si le gain est confirmé.

On compare `jscc` (encodeur causal, section 6) à une variante où l'encodeur est
bidirectionnel, à décodeur strictement identique, sur les mêmes trois sources et
les mêmes niveaux de bruit que l'expérience 1. La classe hérite de
`ResidualSystem` : `encode` et `forward` ne sont pas dupliqués, seule
l'instanciation de `self.enc` change.

In [ ]:
class ResidualSystemEncBidir(ResidualSystem):
    '''Identique à ResidualSystem, sauf que l'encodeur (mode jscc) voit tout le
    bloc avant d'émettre — Readout(cfg, bidir=True) au lieu de bidir=False.
    `encode` et `forward` sont hérités sans modification : ils appellent
    self.enc(...) de façon générique, quelle que soit sa largeur de sortie.'''

    def __init__(self, cfg, mode):
        super().__init__(cfg, mode)
        if mode == "jscc":
            self.enc = Readout(cfg, bidir=True)


def train_variant(cfg, model_cls, mode, ds_train, steps=None, seed=0):
    '''Comme train() (section 6), mais paramétrable en classe de modèle, pour
    ne pas toucher à la définition de train() utilisée par les sections
    précédentes.'''
    steps = steps or cfg.steps
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 1)
    model = model_cls(cfg, mode)
    if model.n_params:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        for _ in range(steps):
            t, m = ds_train.sample(cfg.batch, gen)
            loss = distortion(model(t, m, gen=gen))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()
    return model


p_causal = ResidualSystem(CFG, "jscc").n_params
p_bidir = ResidualSystemEncBidir(CFG, "jscc").n_params
print(f"jscc, encodeur causal          : {p_causal} paramètres, encodeur = N itérations LIF")
print(f"jscc, encodeur bidirectionnel  : {p_bidir} paramètres, encodeur = 2N itérations LIF")

### Balayage — 3 sources × `SIGMA_GRID` × 3 graines

3 graines plutôt que les 2 de l'expérience 1 : l'écart attendu entre encodeur
causal et bidirectionnel est de l'ordre de quelques pourcents, plus petit que le
gain d'encodeur lui-même — il faut plus de répétitions pour le distinguer du
bruit d'entraînement. `SIGMA_GRID` est réutilisé tel quel pour rester comparable
à l'expérience 1.

**Coût** : $3 \times 3 \times 3 = 27$ triplets (décodeur, causal, bidirectionnel)
entraînés à `STEPS` pas chacun, soit environ deux à trois fois la durée de
l'expérience 1 — prévoir 30 à 45 minutes selon la machine. Pour un premier
passage, réduire `FEEDBACK_SEEDS` à `[0]` et/ou `STEPS` dans la cellule de
configuration donne un résultat en quelques minutes, trop bruité pour conclure
mais suffisant pour vérifier que la cellule tourne.

In [ ]:
FEEDBACK_SOURCES = ("poisson", "gamma_k4", "hawkes")
FEEDBACK_SEEDS = [0, 1, 2]

FEEDBACK_ROWS = []
for src in FEEDBACK_SOURCES:
    ds, dse, Db = make_datasets(src, CFG)
    for ratio in SIGMA_GRID:
        cfg = replace(CFG, sigma_ratio=ratio, steps=STEPS)
        d_dec, d_causal, d_bidir = [], [], []
        t0 = time.time()
        for sd in FEEDBACK_SEEDS:
            md_dec = train_variant(cfg, ResidualSystem, "decoder", ds, seed=sd)
            d_dec.append(evaluate(md_dec, cfg, dse)["d"])
            md_causal = train_variant(cfg, ResidualSystem, "jscc", ds, seed=sd)
            d_causal.append(evaluate(md_causal, cfg, dse)["d"])
            md_bidir = train_variant(cfg, ResidualSystemEncBidir, "jscc", ds, seed=sd)
            d_bidir.append(evaluate(md_bidir, cfg, dse)["d"])
        row = {"source": src, "sigma_ratio": ratio,
               "decoder": float(np.mean(d_dec)),
               "jscc_causal": float(np.mean(d_causal)),
               "std_causal": float(np.std(d_causal)),
               "jscc_bidir": float(np.mean(d_bidir)),
               "std_bidir": float(np.std(d_bidir))}
        row["gain_enc_causal"] = row["decoder"] / row["jscc_causal"]
        row["gain_enc_bidir"] = row["decoder"] / row["jscc_bidir"]
        row["bidir_vs_causal"] = row["jscc_causal"] / row["jscc_bidir"]
        FEEDBACK_ROWS.append(row)
        print(f"{src:9s} σ/µ={ratio:4.1f} | decoder={row['decoder']:.5f} | "
              f"causal={row['jscc_causal']:.5f}±{row['std_causal']:.5f} "
              f"(gain {row['gain_enc_causal']:.3f}×) | "
              f"bidir={row['jscc_bidir']:.5f}±{row['std_bidir']:.5f} "
              f"(gain {row['gain_enc_bidir']:.3f}×) | "
              f"bidir/causal={row['bidir_vs_causal']:.3f}×  "
              f"({time.time()-t0:.0f}s)", flush=True)

In [ ]:
show_table(FEEDBACK_ROWS,
           cols=["source", "sigma_ratio", "decoder", "jscc_causal", "jscc_bidir",
                 "gain_enc_causal", "gain_enc_bidir", "bidir_vs_causal"],
           title="Encodeur causal vs bidirectionnel, à décodeur identique")
print()
gains = [r["bidir_vs_causal"] for r in FEEDBACK_ROWS]
print(f"bidir/causal : min={min(gains):.3f}×  médiane={np.median(gains):.3f}×  "
      f"max={max(gains):.3f}×  sur {len(gains)} combinaisons")
print(f"toutes > 1 : {all(g > 1.0 for g in gains)}  "
      "(si non : le bruit d'entraînement dépasse l'effet mesuré sur au moins un point)")

In [ ]:
def fig_feedback(rows, sources=FEEDBACK_SOURCES):
    fig, axes = plt.subplots(1, len(sources), figsize=(4.6 * len(sources), 4.4),
                             facecolor=SURFACE, sharey=True)
    for ax, src in zip(np.atleast_1d(axes), sources):
        sub = sorted([r for r in rows if r["source"] == src], key=lambda r: r["sigma_ratio"])
        xs = [r["sigma_ratio"] for r in sub]
        ax.plot(xs, [r["gain_enc_causal"] for r in sub], color=C_DECODER, lw=2.0,
                marker="o", ms=7, mfc=SURFACE, mew=2, mec=C_DECODER,
                label="encodeur causal")
        ax.plot(xs, [r["gain_enc_bidir"] for r in sub], color=C_JSCC, lw=2.0,
                marker="^", ms=7, mfc=SURFACE, mew=2, mec=C_JSCC,
                label="encodeur bidirectionnel")
        ax.axhline(1.0, color=INK, lw=1.2, ls=(0, (4, 3)), zorder=4)
        ax.set_xscale("log")
        style(ax, "$\\sigma_J$ / écart moyen  (log)",
              "gain d'encodeur  (decoder / jscc)" if src == sources[0] else "", src)
    legend(np.atleast_1d(axes)[-1], loc="upper right")
    fig.suptitle("Voir tout le bloc aide l'encodeur, mais peu — la fenêtre saturée "
                 "reste le facteur limitant", color=INK, fontsize=11, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return fig


fig_feedback(FEEDBACK_ROWS); plt.show()

**Lecture.** L'encodeur bidirectionnel bat systématiquement le causal (les deux
courbes se superposent presque, la bidirectionnelle très légèrement au-dessus) —
un effet réel mais petit, de l'ordre de quelques pourcents, à comparer au gain
d'encodeur lui-même (`decoder`/`jscc`, déjà modeste dans le problème complet).
Il est le plus marqué sur Hawkes à fort bruit : c'est la source dont la structure
de bloc (rafales, $N$ très variable) laisse le plus de marge à une allocation
globale plutôt qu'à une suite de décisions locales.

**Ce que ça dit sur la cause du plateau.** Le fossé entre le gain d'encodeur du
Step 3 (2 à 10×) et celui du problème complet (quelques pourcents) n'est donc
**pas** principalement dû à la causalité de l'encodeur — sinon lever la
contrainte causale aurait comblé une part substantielle de l'écart. La
saturation de la fenêtre ($\sum_i g_i \approx T_c$, section 9) reste
l'explication dominante : elle plafonne ce qu'il y a à gagner, que l'encodeur
voie l'avenir du bloc ou non. Le bidirectionnel ne fait que mieux répartir ce
qui reste à gagner à l'intérieur de ce plafond.

**Verdict pratique.** Le gain ne justifie pas de doubler les paramètres et la
latence de l'encodeur par défaut — mais si le rapport final soulève la question
de la causalité de l'encodeur, ce sont les chiffres à citer, plutôt qu'un
argument seulement qualitatif.

## 13. Limites et suites

### Limites assumées

- **Le décodeur est bidirectionnel, l'encodeur est causal.** C'est justifié (code
  par bloc : le récepteur a tout le bloc) mais ça veut dire que `decoder` et `jscc`
  ne sont pas des SNN strictement causaux en ligne. Un test avec décodeur causal
  est une variante à une ligne (`bidir=False`) et vaut la peine d'être chiffrée.
- **Une seule couche cachée LIF**, feed-forward, 32 unités. Choix de simplicité
  assumé, pas une limite de capacité démontrée. L'écart résiduel à l'optimum dans
  le Step 3 (facteur 1.3-2.6) est le bon endroit pour tester si la capacité est en
  cause — et le Step 3 est assez petit pour balayer la largeur rapidement.
- **`a_max = log 4`** borne la dilatation par écart. Dans le Step 3 l'optimum
  demande une pente locale de l'ordre de 6, donc cette borne est potentiellement
  active dans le problème complet. À balayer.
- **Le réordonnancement de $Y$ est fréquent** (`reorder_frac_Y` monte au-dessus de
  90 % de blocs à fort bruit) : le récepteur trie, ce qui est optimal au sens MSE
  pour une cible ordonnée, mais l'appariement par indice de l'éq. 7 devient
  discutable dans ce régime. Une distance de Wasserstein 1-D ou une distance de
  Victor-Purpura donnerait une mesure moins arbitraire ; c'est une extension
  naturelle.
- **Deux graines par point** dans les sweeps. Mieux que la v1 (une seule), encore
  insuffisant pour trancher sur des écarts de quelques pourcents. Augmenter `SEEDS`
  avant de conclure sur le gain d'encodeur dans le problème complet.
- Toute la comparaison est à $R = 1$. Le PDF le demande pour les expériences
  initiales, mais $R \ne 1$ (plus ou moins de temps canal que de temps source)
  changerait qualitativement la marge de dilatation disponible — c'est
  probablement la suite la plus intéressante.

### Suites

1. **Balayer $R = T_s/T_c$.** Le mécanisme identifié (dilatation à coût d'événements
   constant) prédit que le gain croît avec $T_c/T_s$, puisque la fenêtre cesse d'être
   saturée. C'est une prédiction falsifiable directement testable avec ce code.
2. **Fermer l'écart à l'optimum dans le Step 3** : largeur du réseau, $a_{\max}$,
   nombre de pas, et une comparaison à une table monotone libre apprise par SGD
   (même famille que l'optimum numérique, mais entraînée sur échantillons) pour
   séparer « la famille de fonctions est trop pauvre » de « l'optimisation
   n'aboutit pas ».
3. **Remonter le compandage du Step 3 vers le problème complet** : imposer à
   l'encodeur du système complet la forme trouvée optimale en deux événements, et
   voir si ça bat l'encodeur appris librement. Si oui, le problème est
   l'optimisation ; si non, la contrainte de fenêtre domine.
4. **Distorsion sur la géométrie relative plutôt que sur les temps absolus** :
   remplacer $\sum_i (T_i - \hat T_i)^2$ par une distorsion sur les écarts
   $\sum_i (g_i - \hat g_i)^2$. La question de recherche du PDF porte sur la
   géométrie temporelle relative ; la mesurer directement changerait ce que le
   système est incité à préserver.

### Références principales

- Gastpar, Rimoldi & Vetterli, *To code, or not to code: lossy source-channel
  communication revisited*, IEEE Trans. IT 49(5), 2003 — conditions nécessaires et
  suffisantes d'optimalité de la transmission non codée. Le cadre qui explique
  pourquoi `uncoded` est une baseline difficile et où un gain peut exister.
- Rubin, *Information Rates and Data-Compression Schemes for Poisson Processes*,
  IEEE Trans. IT, 1974 ; Shen, Moser & Pfister, *Rate-Distortion Problems of the
  Poisson Process based on a Group-Theoretic Approach*, arXiv:2202.13684 — la
  théorie débit-distorsion pour ce type de source, y compris sur les intervalles.
- Skatchkovsky, Jang & Simeone, *End-to-End Learning of Neuromorphic Wireless
  Systems*, Asilomar 2020 (NeuroJSCC) — le travail antérieur le plus proche.
- Neftci, Mostafa & Zenke, *Surrogate Gradient Learning in Spiking Neural Networks*,
  IEEE SPM, 2019 — le gradient de substitution utilisé ici.
- Srinivas, Adve & Eckford, *Molecular communication in fluid media: the additive
  inverse Gaussian noise channel*, IEEE Trans. IT, 2012 — le canal de timing additif
  vu par la théorie de l'information.
- Du et al., *Recurrent Marked Temporal Point Processes*, KDD 2016 ; Mei & Eisner,
  *The Neural Hawkes Process*, NeurIPS 2017 — le déroulement événement par événement
  adopté ici.